## 1. 安裝與 GPU 檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch

'''
Stop before model downloads unless exactly two CUDA GPUs are available.
若沒有兩張 CUDA GPU，就在下載模型前停止。
'''
assert torch.cuda.device_count() == 2, '請在 Kaggle Settings 選擇 GPU T4 ×2。'

'''
Keep source code and final MP3 in Kaggle's working directory.
將原始碼與最終 MP3 放在 Kaggle 工作目錄。
'''
ROOT = Path('/kaggle/working')

'''
Locate the official source repository.
設定官方原始碼路徑。
'''
REPO = ROOT / 'heartlib'

'''
Store large model weights outside the Kaggle output directory.
將大型模型權重放在 Kaggle 輸出目錄以外。
'''
CKPT = Path('/tmp/heartmula/ckpt')

'''
Create model-storage directories when needed.
必要時建立模型儲存目錄。
'''
CKPT.mkdir(parents=True, exist_ok=True)

'''
Keep the Hugging Face cache on temporary storage.
將 Hugging Face 快取放到暫存空間。
'''
os.environ['HF_HOME'] = '/tmp/heartmula/huggingface'

'''
Move the Xet download cache to temporary storage.
將 Xet 下載快取放到暫存空間。
'''
os.environ['HF_XET_CACHE'] = '/tmp/heartmula/huggingface/xet'

'''
Disable tokenizer parallelism warnings.
關閉 tokenizer 平行處理警告。
'''
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

'''
Download the official source if it is not already available.
只有尚未存在官方原始碼時才下載。
'''
if not (REPO / 'pyproject.toml').exists():

    '''
    Clone the upstream repository quietly.
    以安靜模式複製官方程式庫。
    '''
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HeartMuLa/heartlib.git', str(REPO)], check=True)
else:

    '''
    Verify the existing source can be updated safely.
    確認現有原始碼可安全更新。
    '''
    assert (REPO / '.git').exists(), 'HeartMuLa 路徑不是 Git 倉庫，請先檢查。'

    '''
    Check for local modifications, including the previous RoPE patch.
    檢查本機修改，包括先前套用的 RoPE 修正。
    '''
    changes = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()

    '''
    Update only when doing so will not overwrite customized files.
    只有不會覆寫使用者修改時，才同步上游程式。
    '''
    if not changes:

        '''
        Accept only fast-forward changes from the upstream branch.
        僅允許上游分支進行 fast-forward 更新。
        '''
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', '-q'], check=True)
    else:

        '''
        Preserve the local RoPE patch and other edits.
        保留現有 RoPE 修正與其他本機修改。
        '''
        print('HeartMuLa 原始碼已有本機修改，為避免衝突未自動更新；建議使用全新 Kaggle Session。')

'''
Display the exact installed source revision.
顯示目前安裝的原始碼版本。
'''
print('HeartMuLa commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())

'''
Install compatible model dependencies without replacing PyTorch.
安裝相容的模型套件，避免覆蓋 Kaggle 原有 PyTorch。
'''
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.0', 'tokenizers==0.22.1',
    'huggingface-hub==0.36.2', 'vector-quantize-pytorch==1.27.15', 'soundfile',
], check=True)

'''
Keep Kaggle's PyTorch and TorchAO when installing TorchTune.
安裝 TorchTune 時保留 Kaggle 的 PyTorch 與 TorchAO。
'''
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchtune==0.4.0'], check=True)

'''
Register the official HeartMuLa package without reinstalling its dependencies.
註冊官方 HeartMuLa 套件，不重裝相依套件。
'''
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO)], check=True)

'''
Make the actual src package take precedence over the same-named checkout folder.
讓真正的 src 套件優先於同名的 Git 專案目錄，避免 namespace package 衝突。
'''
sys.path.insert(0, str(REPO / 'src'))

'''
Give installation checks the same import priority in their subprocesses.
讓安裝檢查子程序也優先載入正式套件，而非外層 Git 目錄。
'''
os.environ['PYTHONPATH'] = str(REPO / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')

'''
Check the critical imports before downloading large model files.
下載大型權重前先測試必要套件是否能匯入。
'''
subprocess.run([
    sys.executable, '-c',
    'import torchao, torchtune, torchaudio; from heartlib import HeartMuLaGenPipeline',
], check=True)

'''
Display both physical GPU names.
顯示兩張實體 GPU 的名稱。
'''
for gpu in range(2):

    '''
    Report the CUDA device assigned to each stage.
    顯示各階段對應的 CUDA 裝置。
    '''
    print(f'GPU {gpu}: {torch.cuda.get_device_name(gpu)}')


## 2. 官方模型與 RoPE 相容修正


In [ ]:
'''
Import the official Hugging Face model snapshot downloader.
匯入 Hugging Face 官方模型下載工具。
'''
from huggingface_hub import snapshot_download

'''
Download the generation metadata and both original-weight models.
下載生成設定與兩個原始權重模型。
'''
for model_id, directory, patterns in [
    ('HeartMuLa/HeartMuLaGen', CKPT, ['tokenizer.json', 'gen_config.json']),
    ('HeartMuLa/HeartMuLa-oss-3B-happy-new-year', CKPT / 'HeartMuLa-oss-3B', ['config.json', '*.safetensors', '*.safetensors.index.json']),
    ('HeartMuLa/HeartCodec-oss-20260123', CKPT / 'HeartCodec-oss', ['config.json', '*.safetensors', '*.safetensors.index.json']),
]:

    '''
    Download only the files required for inference.
    只下載推論所需的檔案。
    '''
    snapshot_download(repo_id=model_id, local_dir=str(directory), allow_patterns=patterns, max_workers=2)

'''
Locate the upstream cache-initialization implementation.
找到官方模型快取初始化的原始碼。
'''
model_file = REPO / 'src/heartlib/heartmula/modeling_heartmula.py'

'''
Read the source before applying the known RoPE compatibility patch.
套用已知 RoPE 相容修正前先讀取原始碼。
'''
source = model_file.read_text(encoding='utf-8')

'''
Identify the upstream cache initialization that needs adjustment.
定位需要調整的官方快取初始化區段。
'''
original = '        with device:\n            self.backbone.setup_caches(max_batch_size, dtype)'

'''
Rebuild missing RoPE caches on the target CUDA device before setup.
先在指定的 CUDA 裝置重建缺少的 RoPE 快取。
'''
replacement = (
    "        with device:\n"
    "            # Restore RoPE caches on the actual compute device.\n"
    "            # 在實際運算裝置上重建 RoPE 快取。\n"
    "            for module in self.modules():\n"
    "                # Repair position embeddings only when their cache is missing.\n"
    "                # 只修復尚未建立快取的位置 embedding。\n"
    "                if hasattr(module, 'rope_init') and not getattr(module, 'is_cache_built', True):\n"
    "                    # Initialize the missing positional cache.\n"
    "                    # 初始化缺少的位置快取。\n"
    "                    module.rope_init()\n"
    "                    # Keep the positional module on the model GPU.\n"
    "                    # 確保位置模組維持在模型所使用的 GPU。\n"
    "                    module.to(device)\n"
    "            self.backbone.setup_caches(max_batch_size, dtype)"
)

'''
Avoid applying the compatibility fix twice.
避免重複套用相容修正。
'''
if 'module.rope_init()' not in source:

    '''
    Refuse to patch an unrecognized upstream source layout.
    若官方原始碼結構改變，避免直接修改未知區段。
    '''
    assert original in source, 'HeartMuLa 的 setup_caches 已變更，請檢查官方版本。'

    '''
    Apply the patch exactly once without touching checkpoint weights.
    只修正一處程式碼，不更動模型權重。
    '''
    model_file.write_text(source.replace(original, replacement, 1), encoding='utf-8')


## 3. 載入模型（僅首次）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import sys
from pathlib import Path
import torch

'''
Prioritize the official Python package under the repository's src directory.
優先載入官方專案 src 目錄內的 Python 套件。
'''
sys.path.insert(0, str(REPO / 'src'))

'''
Inspect any previously cached namespace-only package.
檢查先前可能暫存的錯誤 namespace package。
'''
cached_heartlib = sys.modules.get('heartlib')

'''
Remove a namespace-only import without discarding a valid package.
僅移除錯誤的 namespace 快取，保留已正確匯入的套件。
'''
if cached_heartlib is not None and getattr(cached_heartlib, '__file__', None) is None:

    '''
    Allow Python to resolve the actual package from src.
    讓 Python 從 src 重新解析真正的套件。
    '''
    del sys.modules['heartlib']

'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
from heartlib import HeartMuLaGenPipeline
import heartlib

'''
Refuse a different module with the same package name.
若匯入了其他同名套件則立即停止。
'''
assert Path(heartlib.__file__).resolve().is_relative_to(REPO / 'src'), f'heartlib 匯入來源錯誤：{heartlib.__file__}'

'''
Assign token generation and audio decoding to separate GPUs.
將 token 生成與音訊解碼分配到不同 GPU。
'''
devices = {'mula': torch.device('cuda:0'), 'codec': torch.device('cuda:1')}

'''
Reuse an existing pipeline to avoid duplicating large models in VRAM.
重複執行此格時沿用既有 Pipeline，避免 VRAM 中出現第二份模型。
'''
if globals().get('pipe') is None:

    '''
    Load both model components once without lazy loading.
    僅載入一次雙 GPU 模型，停用 Lazy Loading。
    '''
    pipe = HeartMuLaGenPipeline.from_pretrained(
        str(CKPT), device=devices,
        dtype={'mula': torch.float16, 'codec': torch.float32},
        version='3B', lazy_load=False,
    )

    '''
    Confirm the initial model loading has completed.
    確認首次載入模型已完成。
    '''
    print('已首次載入模型；後續生成請執行第 4、5、6 節。')
else:

    '''
    Confirm that the existing pipeline is the intended implementation.
    確認既有 Pipeline 是正確的官方類別。
    '''
    assert isinstance(pipe, HeartMuLaGenPipeline), '既有 pipe 不是 HeartMuLaGenPipeline。'

    '''
    Report that the existing model will be reused.
    顯示正在沿用已載入的模型。
    '''
    print('沿用既有模型，未再次載入權重。')

'''
Require both model components to remain loaded instead of lazy loading.
確認兩個模型均已載入，沒有使用 Lazy Loading。
'''
assert not pipe.lazy_load and pipe._mula is not None and pipe._codec is not None, '雙 GPU 模型未完整載入。'

'''
Verify that all HeartMuLa parameters are on GPU 0.
確認 HeartMuLa 所有參數均位於 GPU 0。
'''
assert {p.device for p in pipe.mula.parameters()} == {devices['mula']}, 'HeartMuLa 並未完整位於 GPU 0。'

'''
Verify that all HeartCodec parameters are on GPU 1.
確認 HeartCodec 所有參數均位於 GPU 1。
'''
assert {p.device for p in pipe.codec.parameters()} == {devices['codec']}, 'HeartCodec 並未完整位於 GPU 1。'

'''
Show allocated memory for both CUDA devices.
顯示兩張 GPU 已配置的記憶體用量。
'''
for gpu in range(2):

    '''
    Report memory still occupied by loaded model parameters and caches.
    回報模型參數與快取目前佔用的記憶體。
    '''
    print(f'GPU {gpu}：{torch.cuda.memory_allocated(gpu) / 1024**3:.2f} GiB 已配置')


## 4. 生成設定


In [ ]:
'''
Define the lyrics for the next generation.
設定下一首音樂使用的歌詞。
'''
LYRICS = '''
[Verse]
十年生死兩茫茫
不思量，卻自難忘
千里孤墳隔月光
無處話淒涼

[Chorus]
夜來幽夢，忽然還鄉
小軒窗前，你正梳妝
相顧無言，淚落千行
一醒只剩，滿地月光

[Bridge]
料得年年最斷腸
明月夜，短松岡

[Chorus]
隔著十年，隔著陰陽
明月仍照，那座松岡
'''.strip()

'''
Describe the musical style using comma-separated tags.
使用逗號分隔的標籤描述曲風。
'''
TAGS = 'soft,sad,ballad,longing'

'''
Limit the next audio generation to this many seconds.
設定下一首音訊的生成秒數上限。
'''
DURATION = 20

'''
Choose a fixed seed, or -1 for a random seed on each generation.
設定固定種子，或使用 -1 讓每次生成採用隨機種子。
'''
SEED = 42

'''
Set the MP3 filename in the same configuration cell.
在同一個設定格指定 MP3 檔名。
'''
OUTPUT_NAME = '十年生死兩茫茫.mp3'


## 5. 生成音樂（可重複執行）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import time
import heartlib.pipelines.music_generation as music_generation
import random
import secrets
import numpy as np

'''
Resolve a reproducible seed; -1 generates a new seed for this run.
使用固定種子重現結果；-1 則為本次生成產生新種子。
'''
RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

'''
Keep the seed within the shared 32-bit range of NumPy and PyTorch.
種子限制在 NumPy 與 PyTorch 均支援的 32-bit 範圍。
'''
assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'

'''
Seed all libraries before sampling without rebuilding the loaded models.
推論取樣前設定隨機種子，不重新建立已載入的模型。
'''
random.seed(RUN_SEED)

'''
Seed NumPy before sampling.
取樣前同步設定 NumPy 種子。
'''
np.random.seed(RUN_SEED)

'''
Seed the main PyTorch random generator.
設定 PyTorch 的主要隨機數產生器種子。
'''
torch.manual_seed(RUN_SEED)

'''
Seed random generators on all available GPUs.
同步設定所有可用 GPU 的隨機數產生器種子。
'''
torch.cuda.manual_seed_all(RUN_SEED)

'''
Show the effective seed for repeatable music-generation tests.
顯示實際種子，方便學生重現同一組音樂生成設定。
'''
print(f'本次生成 seed：{RUN_SEED}')

'''
Save the latest lyric text to a file accepted by the official pipeline.
將最新歌詞寫入官方 Pipeline 可讀取的文字檔。
'''
lyrics_file = ROOT / 'heartmula_lyrics.txt'

'''
Update the input file on every new generation.
每次生成時更新歌詞檔，確保使用最新內容。
'''
lyrics_file.write_text(LYRICS, encoding='utf-8')

'''
Save the latest style description to its input file.
將最新曲風標籤寫入輸入檔案。
'''
tags_file = ROOT / 'heartmula_tags.txt'

'''
Update style tags before each generation.
每次生成之前更新曲風標籤。
'''
tags_file.write_text(TAGS, encoding='utf-8')

'''
Interpret the filename configured above as a local MP3 filename.
將設定格指定的檔名視為本地 MP3 檔名。
'''
filename = Path(OUTPUT_NAME)

'''
Reject directory paths and non-MP3 extensions.
防止輸出到其他資料夾，並限制副檔名為 MP3。
'''
assert filename.name == OUTPUT_NAME and filename.suffix.lower() == '.mp3', 'OUTPUT_NAME 必須是單純的 .mp3 檔名。'

'''
Use the configured filename for the first generation.
第一次生成直接使用指定的檔名。
'''
output = ROOT / filename

'''
Start numbering at 002 only when the target file already exists.
只有指定檔案已存在時，才從 002 開始加上流水號。
'''
index = 2

'''
Preserve previous music by adding a suffix to repeated filenames.
同名檔案已存在時自動加上編號，避免覆蓋先前的音樂。
'''
while output.exists():

    '''
    Choose the next available numbered MP3 filename.
    依序尋找下一個尚未使用的 MP3 檔名。
    '''
    output = ROOT / f'{filename.stem}_{index:03d}.mp3'

    '''
    Move to the next number if another file has the same name.
    若該編號也已使用，繼續尋找下一個編號。
    '''
    index += 1

'''
Keep the upstream progress implementation for restoration after this run.
暫存官方進度機制，於本次執行結束後還原。
'''
original_tqdm = music_generation.tqdm

'''
Print actual generated-frame progress, even without Notebook widgets.
即使 Notebook 沒有啟用 widget，也能顯示實際逐幀生成進度。
'''
def show_progress(frame_indices):

    '''
    Read the total number of 80 ms frames allowed in this generation.
    取得本次允許生成的 80 毫秒音訊幀數量。
    '''
    total = len(frame_indices)

    '''
    Report roughly every 10% to avoid excessive log lines.
    大約每 10% 回報一次，避免輸出過多紀錄。
    '''
    interval = max(1, total // 10)

    '''
    Indicate that frame-by-frame generation has begun.
    顯示已開始逐幀產生音訊。
    '''
    print(f'音訊生成中：最多 {total} 幀（{DURATION} 秒）', flush=True)

    '''
    Pass each original frame index to the official generation loop.
    將每個原始幀索引交回官方的生成迴圈。
    '''
    for frame in frame_indices:

        '''
        Report completed frames at regular intervals.
        定期回報已完成的音訊幀數量。
        '''
        if frame > 0 and frame % interval == 0:

            '''
            Show the generated-duration limit fraction and wall-clock time.
            顯示生成長度上限的進度，以及實際經過時間。
            '''
            print(f'長度進度 {frame / total:.0%}｜約 {frame * 0.08:.1f}/{DURATION} 秒｜耗時 {time.perf_counter() - start:.0f} 秒', flush=True)

        '''
        Keep inference behavior unchanged.
        保持官方推論行為不變。
        '''
        yield frame

    '''
    Report 100% only if the full configured frame limit is reached.
    僅在完整達到設定的音訊幀數上限時顯示 100%。
    '''
    print('長度進度 100%｜已達音訊長度上限', flush=True)

'''
Record which CUDA devices actually execute the selected model layers.
記錄本次實際執行指定模型層的 CUDA 裝置。
'''
used_gpus = set()

'''
Capture the device of each selected layer's output tensor.
以指定模型層的輸出 Tensor 裝置確認實際運算位置。
'''
def record_gpu(module, inputs, result):

    '''
    Identify the device used by this model operation.
    取得本次模型運算實際使用的 GPU 編號。
    '''
    gpu = result.device.index

    '''
    Announce audio decoding when the GPU 1 decoder first runs.
    當 GPU 1 的解碼器首次執行時顯示解碼狀態。
    '''
    if gpu == 1 and gpu not in used_gpus:

        '''
        Stream the decoding-stage message into this cell.
        在本格輸出區即時顯示音訊解碼訊息。
        '''
        print('[2/3] GPU 1 正在解碼音訊...', flush=True)

    '''
    Accumulate GPU indices observed during this generation.
    記錄本次生成過程實際執行運算的 GPU 編號。
    '''
    used_gpus.add(gpu)

'''
Observe token-generation computations on GPU 0.
監測 GPU 0 上的 token 生成運算。
'''
mula_hook = pipe.mula.codebook0_head.register_forward_hook(record_gpu)

'''
Observe waveform-decoder computations on GPU 1.
監測 GPU 1 上的音訊解碼運算。
'''
codec_hook = pipe.codec.decoder.scalar_model.decoder[0].register_forward_hook(record_gpu)

'''
Start timing the current generation run.
開始計算本次音樂生成所花費的時間。
'''
start = time.perf_counter()

'''
Report the new output name before time-consuming model inference.
在耗時的模型推論前，先顯示本次音樂輸出檔名。
'''
print(f'[1/3] 正在生成 {output.name}，音訊長度上限 {DURATION} 秒...', flush=True)

'''
Use text progress for this run only, without modifying model weights.
只在本次執行使用文字進度顯示，不修改模型權重。
'''
music_generation.tqdm = show_progress

'''
Ensure monitoring hooks and official progress are restored on failure.
即使推論失敗，也要還原官方進度機制並移除監測 hook。
'''
try:

    '''
    Disable gradients to avoid unnecessary inference memory use.
    停用梯度計算，避免推論時產生不必要的記憶體用量。
    '''
    with torch.no_grad():

        '''
        Reuse both resident models through the official pipeline's generation API.
        透過官方 Pipeline 重用已駐留的雙 GPU 模型完成生成與解碼。
        '''
        pipe(
            {'lyrics': str(lyrics_file), 'tags': str(tags_file)},
            save_path=str(output),
            max_audio_length_ms=DURATION * 1000,
            temperature=1.0, topk=50, cfg_scale=1.5,
        )
finally:

    '''
    Restore the official progress function for other pipeline uses.
    還原官方進度函式，不影響其他 Pipeline 使用情境。
    '''
    music_generation.tqdm = original_tqdm

    '''
    Remove the GPU 0 monitoring hook after this run.
    本次執行結束後移除 GPU 0 的監測 hook。
    '''
    mula_hook.remove()

    '''
    Remove the GPU 1 monitoring hook after this run.
    本次執行結束後移除 GPU 1 的監測 hook。
    '''
    codec_hook.remove()

'''
Confirm that generation and MP3 saving returned from the pipeline.
確認 Pipeline 已完成生成與 MP3 儲存流程。
'''
print('[3/3] 已完成音訊生成與儲存，正在檢查結果...', flush=True)

'''
Verify that the models actually performed operations on both GPUs.
確認兩張 GPU 都實際執行了模型運算。
'''
assert used_gpus == {0, 1}, f'雙 GPU 運算驗證失敗：{used_gpus}'

'''
Require the generated MP3 to exist and contain nonempty audio data.
確認本次成功產生非空的 MP3 音訊檔案。
'''
assert output.is_file() and output.stat().st_size > 1024, '找不到有效的 MP3。'

'''
Report actual memory allocations on both GPUs after generation.
回報本次生成後兩張 GPU 的實際記憶體配置。
'''
for gpu in range(2):

    '''
    Show that the model remains resident for the next generation.
    顯示模型仍駐留在 GPU，供下一次生成直接使用。
    '''
    print(f'GPU {gpu}：已驗證運算；VRAM {torch.cuda.memory_allocated(gpu) / 1024**3:.2f} GiB')

'''
Report this run's generated file and elapsed minutes.
顯示本次生成的檔案名稱與實際耗時。
'''
print(f'MP3：{output}；生成耗時：{(time.perf_counter() - start) / 60:.2f} 分鐘')


## 6. 試聽與下載


In [ ]:
'''
Import notebook audio playback and download links.
匯入 Notebook 音訊播放與下載連結工具。
'''
from IPython.display import Audio, FileLink, display

'''
Preview the completed MP3 in Kaggle.
在 Kaggle 中試聽完成的 MP3。
'''
display(Audio(filename=str(output)))

'''
Provide a download link for the generated MP3.
提供生成 MP3 的下載連結。
'''
display(FileLink(str(output)))


## 7. ngrok 與遠端生成 API 啟動

In [ ]:
'''
Dependencies are available in the Python standard library.
只使用 Python 標準函式庫處理遠端服務與檔案。
'''
import zlib
import math
import re

import queue
import copy

'''
Use secure randomness when the API requests seed=-1.
API 指定 seed=-1 時，以安全的隨機來源產生實際種子。
'''
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

'''
Expose only the inference bridge, never the Jupyter or ComfyUI management interfaces.
ngrok 僅轉送推論橋接 API，不公開 Jupyter 或 ComfyUI 管理介面。
'''
REMOTE_PORT = 8765
REMOTE_SERVICE_ID = '06_HeartMuLa_音樂生成_Kaggle'
REMOTE_KIND = 'music'
REMOTE_FIELDS = ['task', 'lyrics', 'tags', 'duration', 'seed']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xadZ[O\x1bY\xb6~\xcf\xaf\xd8R\xab\x05\xe4\x04\x07H\xf7\x9c\xee\x9ca\xa4Lw\xab\x15\xcd\xe9\x04\xe52\xd2Qkd\x15\xf6\xb6]\xd3v\x95\xa7\xaa\x0c\xf1AH\x90\x0e\x01\x02\x044\r$\\\x12r\x8fs\x01\x92t:!\x84\x90\x1f3\xae\xb2\xfd\x94\xbf0\xdf\xdakW\xb9\xec\x90\xd6y8y\x08P\xae\xbd\xf7\xb7\xee\xdfZ\xdb\x9f\x89\x93\x85\xa2\xedx\xc23\x0b\xa6\x95\x15%\xcf\xcc\x9b\x9e)]\xe1\xd9\xc2\x91\xea\xa3\xac\xb4\xa4cx\xa6m\x89t\x89\x7fI\x1c\xfaL\xf8\xb3\xdb\xfe\xc4\xfdze*X\xb9\xe8\xbf\xbe\xefO\xbc\xae\xbe\xbd\xdf\xb8\xbd]\xbb\xb7\xdb\xd8\xf85\xa8\\\xac-n\x04S\x0b\xf5\xb1kx\xe1_c\x17\x0f\x99\xd1I\xf2\x10\xd6\x9fH\xa5\xa4\x8bsrR\xd8\x99\x8c\x992\x8d\xbc(\x9aE\x997-)\nv\xba\x94\x971\x10\x19\xc7(HQt\xec\xacC\xabL\x0b\x0bMW\xa4d>\xaf\xc0\xcc/\xfb\xfb\xd7\xfc\xad\xeb\xc1\xf2\x1b1\x10\xee\x12Tn\xd7~\xbb\xf4ao\xb6\xbe\xf5Kcl\xc1\x7f3\xd6\x18{\xe1\xef>`\x94\xfez%X\x7f\x1a\xdc\xda\xab\xee\xcc`Y\x0cbN\x1a\x8e\x977\x07\x13!\x1e7Q(\xb9f*\x19S\x85\xe1\x8a\xf6g$\x95V\xa7\xeb\x19V\xdap\xd2\x02?\xc4\xa9Ra\xa0,\x1c\xfcj\x17\x84U*\x0cJ\'T\xaa\xed\xb8M]\x8a\x81\xb2\x97\xc3\xd6\xf5\xa9I\xbd\xa6\xb6z\xa9\xb1Z\t\x1em\x04K;\xb5\xc5;\xd0\xa8\xbfR\x89\x01\xe5=\x9b\xc7\x1a\xc2\x95\xa9\x92#\xf1C\xa6\x85k\x97\x9c\x94\x14\x19\xdbQZ\xee\xee\x15v1\xb2\xde\xde\x12N\xf4\xb7\xa6\xfd\x89JtJu\xffr\xb0\xbb\x00}\x05\xf3\x0b\xd5\xfd5ZQ\x9f|\xec\xcf,\xf1\xd9\xb5\xca\x96\xbf\xb9\x80\xb7\xfd+\x1b\xf5\x9f\xdf\xc5p\xe0TGz.\xf6=[\xb6R9\xc7\xb6\xcc\xff\x95,C\xf7\xa0\xe1\x12\x16\xcfN\xe5\x0c\xd73S\xa2hx9W\x0c\xe7$\xd9P\x96\x85\x01\xbc%\xbcC\xb0\xea3\xf7\x83\xab\x95\xfa\x93\xe5\xe0\xb7\xf1Ze\xa6\xfa\xee}m\xb1\xa2\x95\xc1\x18\x1b\xe33\xb5\xadk\xc0X}\xb3\xe1/\xcc\x06\x9bp\xc1M\x7fk\x95\xc1\xc50A\xd1\xc52\x99\xc9*\x92a\xceH\xd7\xce\x0fI\xa8\x08\x1e\xe5\xc0\xbdR\xe6`\x9e\x15\xf5_$\xa86\x08<\xdf\x10\x96\x1cf\r\xb2\xea\xe0gNIi\x8d\xf1\xf8k\xbb\xd1\x81\x8d\xc9\xb9\xda\xfc~\xed\xb7\x85\xe0\xe6\xfa\x87\xbd5l\xe4OO\xd6.\xee\x92g=\xbd\xcd!\xc0\xda\x0b\x96\x9f51\x9e9\x7f*y\xf6\xbb\xef\xbe\x15\xfd\xa1\xee\x12d\xcbA\x99\xb7\x87;\xfb\x0e\x1f>\xd6\xd7%\xcc\x8c\xe0W\xfa\t\x9f\xcc\xbb\x12n\xefu\xd2\xb3.\x92\xe8/R\x16\x95a\x15\xd4a\x13@-\xfe;\x07\x95\xa6\xc5\xb1\xbe\xeeAS\xf9H\x96BLk\x91\x1cr\xa0|\xcevR9\x92HK\xb1\xb2\xe0O\xbdB@\xe8\x97\xc8\x03\xf5K\xc2\xbf1\x19,n\x07\xf3/a\xf8p\xcf\xda\xf6%\x7f}\x8e\x041\\WB\xd9=\xe2\x8f\xfd"\x12\xea\x8fB\x89pDt\xa8?\xfd\xf7\x13\x8d[S\xd5\xb7\x88\xb2=\xbcY\x9f\xfcU|\xd1\xf7\xf5\x17_\xff\xe1?\xfb\xbe\xfe\x92\xbcmj\x196\x84\xd2 \'\xf6\xec \xe1\xce\x92LF>/\x10\x86\x8e\xe1PF\x1a\x940\x07\xa43\n\xc5<\xe5*\x92\xd8.A@9X2\xf3izD\xd2\xe7m#\x8d\xb5\xc8 P\x19\x89\xc8\x0e\x85\x0c\x11T\xee\xfa\xd3s\xec.\xecL,=9\xd3\xce\x1c,\t\x13\xf9owkOf\xfc\xd7/\xea{\x08\x8d\xfb\x90\x19I\xc4\xbf9C\xb2r\xb4%H\xdd\x9d\xa1\xac]!VV\\\x1bF\x9d\x9a\xf8\xe0\xb8\xb7\x86\xf1\x1d9\x84UL\xfc\xee\xf6$Y\xc1\x80\x81C\xb3\xe8t\x12\xe5\x11\x15=\xbcw\xf8\n\xc0Ww\xde\xd6\x1f\x8c\x7f\x9cC\x9a\x07{\xca\x13\n\x86U2\xf2\xc9\x83\xcfn?\xca\x15\x94\xff`\x1bc\xc80\xf3\x06\x85\xd1\xf7\x03\xe79\x91\xc5d\x0c\xa6\xc7\x82\xf5i\x7f~\x9bB\x18/\x1c\x9c\xcd\xda\x91\xa4Ji#\x0e\'\x89\x83b\x90\x08S\xce\x1eV\xfa\x90\x99\x8cLy\xe6\x90l\x06+b[\x1a\x9e\x82\xa4\xb2sw,c#\xb6=\x05R\xa7\xff\xed\x07\x8d\xd5\xb5\xc8\x03P\x01\xaa\xfb\xef\xfdM \xdb\xe0\xa0\x86,\xd5\x9d1\x14\x90\x96r\xa6D#\xb8E\x87\xa21\xd3\x11\x8ft\x05\xe4\xc3\xde\xeaH\x08x\xb4\x83!\x1b\x00\xa9\x9c\x932\x8c\'\xf2e\x07\x89\xd0\x93\x17<*q\x86\xc8\x98\x00l\xa0 \x16=\x082X>\xb8**\x05?\x9b\x0c\xd6\xc7\xe0\xa8\xc1\xe6l\xfd\xd1M\x7f\xfb\x89J\xe3\xadu\x0f:\xafo\x8d\xc1\xf1\xc8}\x97\'\xfd\xcdk\xc1\xe3E\xc2\xac\xceu\x93\xea8\xc4\xeb\xe9\xd3\xe7\xc4Q\xd1\xa1\xca]\xa1\x947\x92\xfcy\xc2\xbb\xe0\xa9 <_L\x03\xaf\x02cZE\x84\x9aZ\x08U\xca!\xe9\x94U\x9al\xeaW\x85\xda\xf6|\xa4\x0bT\xfd`\xede\x84\x14\x08\xa0\xe5\xda\x9d\xdd\xea\xfb\x1b\x9cFY\x0e\x7f\xe2\xa1\xbf\xf5\xa6\r\\b\xd81=\x99$\x05u\xfe\xf7\xff\x9c9\xf9\xcd\xd9#BZ)\x9b\x82\xbc\xbf\xa3\xe4e\xba\xbf:X\xb1\xaeW\x06\xc2\xb4tS\x8eYd\xab\xdb\xc2\xf4\xdc\x18\xfeV-\xae\xbdh\xdc\xa9\x04\x95\x95\xda\xf3{\xac\xcb\xfa\xde\x0e\xfe\x07\xda\xe0\xf6\x94\xf2J#\xfb;\n\xa3O\xdb\xd5\xc5\x18\xe8\x930\x1dH\x03\xd1\xf8iMU\xdf\xcc =he\xc5\x00\xb5\x1c\x1fW\xc9\xb9\x13\xdf\x7fJ!\'-O:E\x14\x14\xa5\x15Zh\x11qJ\xd9V\xc6\xcc\x96\xa8,\x18\x836\x95A\xaasy;\x05\xf7\xfaa\xe0X\xf4\xa2V\x8e\x0e\xe0[{\xc1\xec$\x95:\xb8\xd1\xe3E\x7fa\xae\xfe`\x99k\x9b\xbf\xfeL\xad\xe3\xc7\x044:\xaa_\x0c\xa0\xc0w\x9e>\x7fn\xe0\xfc\xb9\xe4\xa9\x13?|\xd7\xc5\xf5\xf7\xef\x08V\x916\x1d\xfc\xb0\xe1=L\x03\xa8\x1cY\xb6\xd5M\x9bA4i\xb9P\x10G\xe9\xf5\x17\xc1\xe6\x1d2\xc7\xe4\xae?\x05?yU}\xbb\\\xffu2X^\xf1\xef\xed\xab\x9c\xfd@\xd7\xad\xe9m\xc6A%\x04\x1b\xc5\xeaR$\x17C\xeb\x171X\xea\xec\xe8\x05\xb7\x84p\xbb\x90@\xfd\x95Ng\x17\xbd\xda\x91(\x14\x8fu\xa0\x88\xc5\xd7p-\x0b\xaeo\xfb\xcb[\xb5\x97\x8bT\x13\xe9\xb5\xa6"\xd8\x13\\\xf6\xca\x98\xda#\xfd\x84t,c:\xae\xd7\xe6\x15\xb5\xa7O\x91r\x9a\xc4\x01.q\xf5\xbe\x8e\x96VK\x90\x8c(~\xe4\xd4\x91_\x86G\xa8\xc0\xf0\x0c\xe6@ \x9bT\x19\r\xd4\xe8\x9e>\xc4n\xbe\x1cQ/\xf8\xa8\x93\x95:\xaa\x8d\xbc#\x8dt\x19F0u\x9a\xf4\xe7\x1f#\x7f\xf3\xb9\x1c\x0f(\x8b\xfe\xe6u\xe2\xcd+\x17)e\xa2\xa6\xed\xdfT\xfb6\x96g\xfc\x873\xfe\x95[\xd5\x9d+\xa0n\xc1\xb3\x97\xf5\x95\rE\xc6\xac\xb4\xbc\x00\x88}\x04j\x00\xdc]:C\xc4\xe3\xe5\x90i\x974\x85\xa6\x84g\xa4U\x01\x07\x81Uv\xd0\xcc\x1f\xb9<\xa6\xba\xb0\xc2@\xfc\x8f\xf10Qe\x04\xb5\xd7\x15\x1c\x0f\x84\x8d\xf1\xf7\xfe\x04\xbc\xf6r\xfd\x97\x19\x7fb\nQF5H\xa5s\x027\x9cS\xf9L\xa91\xc1\x82wv\x1d?$\xf0\xef3\xf1M\xce\xb6\xb5\x19-J\xd4\xcdb\xc7Z\x05\xb0\xd6\xc8\xe1e\xd5\xfd\x1b\xfe\xee\xbc\xffl&\x98\xdeGc\x01s\xfac3\xfe\xb3\xd5`\xfd1\x1b\x92\\\xa65th\xe1G\xb6\xec\x18i\xba\xa6\'\x0b\xa3\xc9\x11\xa5\xca\xe3=\xc7\xd2\xa3\xca1\x0f\xe9\x03\x7f\xa0p\x86\xba"\xa0\xba\xc3\x00}4,\x1bO\x1d60\xd87\xb3Cr\xc28bp\xee\xfa\xa3\x17\xac3"\xd5\xaf_0PJ\xd7o\xf6j\xbb\xb3m\xc2\xf0\x9b!p6\xf0\x7f\xf4\x8b\xde\x16NZ*\xba\x1e\x1c\xaa\x10k\xd9@\x89dAZ\x1eWc\xae\xd6h\x0b\xc2~*\x83\xc4\xd5B\xb6\x83\xd5\'d]U\xdb\xb8o\x03u@\xb8\xabb\xbd\xc7e\xd7\xdfx]\xbf=K\xf4\xfb\xc6\x86\xbf?\xdb\xb8x\xc9\xbf\xaa\xa0\xd9\x8e\x995-p\x08\xef\x1f\xe9\x02\xd4\xda\xde\xab%\xe89\xfb$\xca8j\xaf\x07\xc2\x115\x01\xe9\xee\xd6~\xf3\x08U<+b\x9c\xa7lO\x0e\xda\xf6Ox\x90\xceJ\x1d-s\xbfBm\xcd\x8f\x82\x17\xff$\xf6\xb3\xb4A\xec\x87\xdf\xe3\xa6\x05\x9dS\x9c\x80p_\xca\xf1\xceR\x12\xfc\xb4\xcc\x80\xc6\xdb\xc3\xc9\x10A\xa7\x02\x94\x84\xb6M\xb4\xcd\x91\x93\x9eA\xccr(\xdb\x1e\xf0k\xd3\x83\xec\x7f\xd5#\n.w\xcd.q5\xa4\xb5t\xd45\xc7\x93\x0e\xef\xc3\xfd\xb3\xd6\xe9\xc4x\xbd\xf2\\g 8+\xb6\n\xb6\x9f\xd4\x1e\xfe\x13\x81S\xaf\\\x01Z\xf0\xb7\xc6\xe4|\xe8\x01|t\xbf\x80\xb7\xb6\xa1<\x14\xa1\xe4n\xd5.es\xc8?\xcc\x1ez{>W\xe4g\xc86\xd3\xc8:4\x0c *\x97\xb7\xb3\x82[\xee\x10\xda\xbd\x87\xb5\x97\x97P/\xd5\n\x7f\xed\xa6\x7f\xeb9\'\xc9f\x80\xab"\xd1\x18\xbf\xea\xdf[\xad\xbd\x1ck\xcc^jz\'\xbcjH\xc1+\x18\x17:{\x8fh\xb4G\x8fb\xb7\x08\xdfI\xc2\xcbL\xc7\xd0\xa3\x86\xee\xc1\xb2\xf6\x81\x18\x83\xa4\x10\x1a\x94\xd9R\xa46m\xc8\xd7/8\xff\x85\xb6$j\xcb\xca\nq\x84d\x91\x9f\xea\xb2\xbf\xb3\t\xaa\x08\x1e\x02\xd4bD\xe1\x1a\x15X\xffaoj\xe4\xdb\xf3gN\x9c;y\xfa\xd4\xa8\x80\xde?\xecM\xa3\x0ce\xf2%7\xd7\x7f\xce)\xc9\x08\xf7\x00\xaa\x1c\x93\x8b\xd0\xdd\xf5\x9c\x84\x83R\'\x85\x88G\xc6$\xc9\xdbv1R0\xc8\xd0\xf6<e\xa9\xab\x1b\x94\xc4!\xc1\xcb;\xfe\xdeRu\xf7\x1ei[\xc5\x1f\xfc@\xf3\xdf\xf7/\xfd\xf5\xa9P,\n\xe2\xf0D\xd1b{v\xd0\x16\xf3\xa7l\xca\x00*\xa1k\xb7\xa4\xde-\x0b\x1a\xe5Dv\xd2V\xd7\xc0Pw\xd67\xd8\xe2\x94\xea\xb7f\xd9#?\xe1\x87\xca\xde\x19\x8d\xe7Oh4U}W\x7f}\x1es\x84~\xd1\xd3\x04\xc7\x07E\x1dE3\xfe\xc3I\x17|\xb1`*\x9fHq\x9a\xc2\x9e\xc3\x88\xa7\xee\x14\xc8\xd3Oj\x98\x95h\xdb\x8d\x9dBG\xf4\xd2kD4J\x12\x98\n!W\x01Ny\xe0\xed}\x7f\xfe\x8anA^\xfd\x02\xd7E\tk,\xff\x12\x97\xa5\xc5o\xd4>\xbc\\\x8c\xb0PG\xd9\x97\x8f\'z>\x1f\xfd\xf0n\x1dA\x12~rX\xf4$z\xbe:\x9e\xe8\xcd\x8c\x1ems\xa5w\xeb<\x90\x83\xc3\x11\xf4\xa2t2\xc9\x94]"\xed\x80\xf5t\xd3\xe0\xca\xf1\xb0eF\xbd~\x90\xdb\xb1\x90*\xd1\x9bV\x06u\xd0JIDE\xce@EwD\xc9J\xe5h\xe0\x90\x8e\x1b\x12\x94?\x98\x1dgO\xe2n\x9c\x92\xf6\xc5]t\xdd\xf5\xad+q\x99\xcb\xa6\xcck\xa3\xb5%\x8f\xde\x1eD\xbf\xa2/\xb0\xb1\xe2O%t\xa0q\x82\xa5DgsQ\x11\xa1\xb8\x08a\xc0\x97~\x9e\x00O \x17Zz\xd9\x18_\x04\xa1\xd4c\xa3VwbK\x91-\x94\x11\xd5\xa9\xad1\xdcj\n\xfa\x1c:\xa5\x0c0\xbe\xc8\x1b\xc5M\xde\xae?\x92&e;p\xa1\x9c\x89\xa8\xfd\xe6\xfc\xb7\'\xd0\xac\x0cQ\xc0\xe8\x12D\xe9\xf1\x82L\x95t\xdb\xe5\xca<\xe8r8\xce@\x97S\x96<1\xacW\xae#\xcd\xe9\x84\xad\xdc\x88K\xa1\xa6kjf\xe1?\xbfG\xf9[\x9dR\xbf{\xa3\xf6n\x8bD\xa1Q[2[\x04\xf9\xa2\xd9\x93\xd7\xa9`}c\x14=\x9a\x1b\xd2\x99\x0c\x88\x8a\x88J-\x11\x04ux\x87\x1bR\x15\xe2\xea<v\x803\x7f|,\xe7dqN\xbd\xa5\x8fG\xf7W\x7f<\xa7\xeb\x9e\x9a\xe3U\xdf\xcdiXT\xf1\x1c\xa5\x1c\x02\xd7\xc9\x03\xe0#\xdc\xb7\xa1\xfe\xa2\x06\x96\xf2^T\xf7N\xa6A$\xccL9\x0e\x98\x04\xe3\xde\x19\xe6g}\xd9\xc5\xdf\xa9s\x8c\x96q0\xa6&AS\xe3\x8a\x16\x9e\x03PP\x18\xc3H\xf0\x89\t\x95fC?=aY\x88#\xe00Ji\xd3\x06(\xee\xce\x9al\x9b\xf6\xec\xe5\xe7\x8a\x93\x11\xfd\x07\xd5\x89\n]m\xe9\x95~\x87\xd4\xf7\xf0n\xed\xce\x9e\xbfRi<X\x8exN\xe4\x97\xfcimf,\x98\x98\x89J]\x861\x82\x89\xa9,E\x7f\x80\xffQf\x8eL\x1e\xcf\xcag\x99\x9f\xb1\x02\x19k7\xa2?+\x05r\xb3K?\xe1\xeev|\xcc\x1e\xcb\xcc\xe1\xe8\\\xb7gsc\xa0?\x11:\x8e\x03\xc6\x88_\x82\xf1\xedx\x88\xeb0\xfa\xb1\xef\xe8\xb1\xbfiy\x83\xcd\xbb\xd8\x91\x17\xf0\xe2D"qp\xc9;\x91J\x95\xa8\xf3\xf6X\x9f\xba\xda\x08{P5\x16i\xba\xa1\xe0A\xe0\x814\'\x1e5:A\x8f_\xadUf\xe2\x11\xc4\x0eq\xb0\x13D\x8aL\xa0_\xe9\xc4/*xN\xf3\xe1H\xc7?I+>q\xa2\x8aWb\xbe\xabff\xb4a\x8f\xea\xf3\xd6\xee\x06;O\xf9oA\r\x0bNS\xabE\x88\x8a0\xd0\xa9j\xca\x90#B\xd9\xaf\xe6@\tz\x90 \x17"\x96\xd9\x93\xcc\x81\x00&PE\xd1\xbbH\'\x89z<l \x82hAg3\x9aZP\x0e\x1bC4\x95(t\x87\x9ex\x10\xca\xde6\x94\xbd\x1ae\xdc\xb4M\x90\xb4O\xaa\x05\xa5z\x92\xd0\'$\xdc\x94\x81"\x9fT1\x19>\xfc\xb1\xe7o\xff7\xdc\xdc\xc9\xea\xfb)\xd5T\x97\x1c\x14\x9d\x96{)\xdd10\x0b\xabW\xa6\x80\x8b\x8d\x1c\x1f\xdd\x05\xd3c\xf5+\xcf\xeb/\xdeR\x13\x1dU[U\xf0\x00\xfb\x80r\xc8\xf9\x9a/\xadTs5\x1c\xe6>\xd5\xc8\xeb\xe1\x0e-\xecF\x1drK\n!\'\x9e\xa82\xaa\xd6\x00\xbe\xad\x8an4M\xd6#\xe9\xe99P\x01\xf4\xa5\x1c6q\xc4\x1cW\xcd\x161\xe4\x01?\xf6R\xd4p\xbc\xe8\x91\xe3\x88\xee_\t\xd3(Q\xe2\x8f\xaa\x90h\xe5\x00\x07\xc4\x96\x9eXP\xf7\x18\xf5k\xf1\xbb\x0fUx\x8fD\xfd\x0fdD\xeemJ;,\xcdl.\x1a\x1a\xe8\xe4\x10&-=\xbePc\xc8\xf8\xf5\x1bO\xdd\xab\xef\xb7\x82\xc57Z+O\xaf4&\xe7\xd8\xeb\x0f\xe8\xd6\xa8^\xc5["\x82\xfd\x1d\xd4\xee\xd0u\xa1e\xa2\x93$D\xe4B<Vj\xceOC\x91\xe8\xa6\x89{Nd\n\xeaA\xd1\xd1cy\xb3}\xd3v\xb9\xf7<X\xd2WL4=WM\xe5\xc7}hu\xe7A\xed\xe1\xdb\xc6\xca=\x1d\'t\xb2\x1a\xd99\xe5\xb0L}k\xbajb\x90u\x8c\xb4\t\x9fu\x9b\x1dO\xc9\xb2$\xf5<\x06Z\xa1&\x8f*\xc8\x02\xcd\xc6\x90g\xa2\x8a5\xbeN\xea\xbb\xb3\x8ds\xd9\xb3\xa3\xb6G\xdf\x94\xc1\xafT\xc7\x01m\xfa\xef\'\x00\x98\xcaG\xe5zp\xf9U\xe3\xc9"\xd6\xc6(2\x19P\xf0\x9c\xdd\xb2\x93\x84*\x9cw\x84\\\x0b\'\x8bA\x1boAO&\x15Y}\x91\x02WP\xed\xdb\xc1\x93\xe9\x8ex\xb2\x15\'\x06N\xc6\x8bEcl\x9eZ\xb3\xb6\x195\xcd\xd8\x17+D\x9c\x1e.\xd4\x96V(\xb7\xac\xad\xa8\\\xa3Y\x04\xb3}n:\xa6&9\xe5\xb4\xd4\x11\xec\xd4\xd9B\x96G:x\x84\xdcq\x1c4\xd6\xe9\x8c\xcd\x93\xbb\x8e\x88\x0e\x9a\xa4\xeaO\xa2\xa1j\xd7\xe8\x91\x96\x1d\\\xe4\xc6$\r&\xfb\xe95\x0e\xac\xae\xd6W\xd0F&U\x85O\xa2\xdf\xcdz\xb9d\xc1\xed\x0f\xc3\x0b\xc4\x1b\x84\xb0\xa7u\x81\'\x0b\x8a\x85\xc0\xd1\xfa{\x13=\xd4\x81\x16\x7f\xea\xff\x12\xbf\xa42\xd9$\xa5Fz\xfeesQ\xd7\xa1\x0c\xb5r\xf9r\xb3\xc9W.\xdb\xa6\xfa(NA:\xa2\x89\n\xcf{\xa2ktX3\xa2\x17\x1f\xbb\xb1?\xf9\xce\xdf\x9b\xe78\xf4\xdf=ol\xbc\xe7!k\xd3H:v\xc1\x9d\xef\xfc\x1c\xea\xfeS\xb1\xd92qi\x92\xf7\x82\xad\'\xf4\\\xec\xda"\xf5\xa3\xb1\x0f/;p\xb8\xc3\xc1\xa6\xf7\xa1\x86\xb45\xec\x18\x9b\xae\x96(-tpg\xd7\'\x80\xf4\xfe?\x01\xe9\xfd\x04\x90fE\x8c!\x01\xcd\xa6v\xc5)\xf0\x9c!\xfee\x02\x8b\x87\x89\xf0?\x02\xe4Hx\x8b\xa5z\x1a\x9b\x19Z\xfc\n\x88Yt\xec\xb6\'j\x8d\xc3`Q{\xf9\x97h8\xca\xf7\xe8\xcd\x12\xd2\xf1\xe31\xaa \xd1\x9a\xf8@\x02+y\x11M\xd8T\x89\t\x1e\xdf\t6\xee\xf3\xed\xf6\x81e\xe3\xaf\xd2a\x16np\x8d\xd4\xc9"\xeae\xa8\xa0\x82kP\xba\r\xa9\xb8\xe2\x18*\xc5\x84\x97\x86\xba-\x98x\xe4\xef\xfd\xa6\xb4\xda\xf8\xf9]\x9c\x91Uw/\xc7\xb9zl\xbc\x1f\xebe\xfa\xc5\x08B\xaaw\x14\x10;\xc2L\xc2\xef7\x1e]\xaboN\x87I}u$Z4\xda\xc1\x15\xfe\x1f%SGW\xd4\xff+\r"W\xab\xa1\xb02\x0f*\xbcG\xf7\xaf\x96m!\xa0\xbdrH\xf3\r\xcfh\xca\xa0\x1b\x8b\xa9\x05\xff\xca\x86\x1e\x03\xdd\xb8Y{\xb4\x1b\xce|Y\xdb\xcd\xcb%-\x86\xae\xe0&\'%\xf4\xe1\xaa|\xf1C\xf0\x13\xf4h\xf8\x91t\xe9;\x15\x7fB\x86\xe9\xfb\x02\xe9LMd\xe7\xd0\xc7\xd2\x8c~iJ\x9f\x10^@h\xda\xa2\xc7\x9a\xba\xaa\xd0\x180u\x80\r\xb4\xef\xb7^E\xf0\xd8%N\x95\xe1\xf91\x1b\xd1\xd7@\x94\x8d\xa2R\xd3\x98\x08{9\xcaD\xd4~@]\xea\xdb\x07\x9d}Q\xe3\xa6G-qw\x81\xbb\xd3\xcd\xb6\xdb,9\xe1\x15\x89\x1ag\x7f\xcc\xe25]R.Q};\xc7\x05\x84\xbe\xbc\x00\\\x94\xcd\xf6\xd7xV}\xe0=J\xfbD\x8ed\x19\x01XP\xa7U\xaaF\xcaY\xc2/\x99\xac\xfd\xf5\xcc\x89\x1fh6\xd7\xbc\x99V\xaaLjUJn\x02\xc4Qe\x95\xc3\x87\x8f\x1dO\xf4eF\xc5\xf7\xe6\x9f;Z\xd9#g\x95f\x99\xd4\xb7\x1a\xca\xd02o\x14\xa9u\x05}D\xdb\x1f\xbb\xa1n\xf9\x0e\x89\xba\xf8\xa1;\x8f\x85\xb9Z\xe59\x85*k?\xfazU(\x10\xbc\x80\xbc\x9c\xdd\x07R\xad\xc5\xbf\x87E\x9ft\xfe\xce\xe4\x87D\xf9C\x0fK\xe1O]n,\\\x87 \xff\x06+\xf0T\xc0').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['music']
REMOTE_MAX_IMAGES = 0

'''
Keep job directories isolated between different notebook services.
各 Notebook 使用獨立工作資料夾，避免不同服務互相干擾。
'''
REMOTE_ROOT = Path('/kaggle/working/remote_api_jobs') / REMOTE_SERVICE_ID
REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('REMOTE_ENGINE_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=32))
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(2))

'''
Results remain available for up to 24 hours, subject to an 8 GiB storage limit.
結果最多保留 24 小時；若超過 8 GiB，會提前清除最舊結果。
'''
REMOTE_RESULT_TTL = 24 * 3600
REMOTE_RESULT_BYTES = 8 * 1024**3
REMOTE_MIN_FREE_BYTES = 2 * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = 2 * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'Kaggle CUDA GPU 不可用；請啟用 GPU 並重新執行模型初始化 Cell。', []

        '''
        Use the model's actual GPU requirements, not the number of queued jobs.
        依各模型原本的 GPU 配置檢查，不因佇列工作數增加 GPU 使用量。
        '''
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [0]
        else:
            selected_gpus = [0, 1]
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]

        '''
        Validate resident PyTorch pipelines and their designated CUDA devices.
        檢查 PyTorch Pipeline 已初始化，且核心模型使用指定 CUDA 裝置。
        '''
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != 'cuda:0':
                return False, 'Qwen Transformer 沒有設定為 cuda:0。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, 0), ('HeartCodec', pipeline.codec, 1)):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            '''
            Query the actual ComfyUI process; a running HTTP server alone is insufficient.
            查詢真正的 ComfyUI 裝置資訊；HTTP 服務存活不等於正在使用 CUDA。
            '''
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐顯存）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task

    '''
    Confirm that the requested video task matches the weights already loaded.
    確認影片模式與目前實際載入的模型權重及量化設定一致。
    '''
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 程序已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))

    '''
    Check numeric types, finite values, and limits used by the generator.
    檢查數值型別、有限值與生成程式的實際有效範圍。
    '''
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]


'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return

        '''
        Reuse the same job ID for retry attempts after an uncertain response.
        網路回應不明時，重試會沿用同一個工作 ID，避免重複生成。
        '''
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)

            '''
            Reject invalid parameters and incomplete model configurations before queuing.
            在排隊前拒絕無效參數與尚未完成初始化的模型設定。
            '''
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):

                '''
                Ignore missing seeds and validate the supported 32-bit range.
                未提供的種子沿用 Notebook 設定，並檢查 32-bit 範圍。
                '''
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')

                    '''
                    Translate -1 into an explicit random value before inference.
                    推論前將 -1 轉成明確的隨機數值。
                    '''
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)

            '''
            Save supplied seeds for reproducibility in the job status API.
            在工作狀態 API 中記錄請求所使用的實際種子。
            '''
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        '''
                        Point R2V to this job's uploaded reference images only.
                        讓 R2V 僅讀取這筆工作的參考圖片資料夾。
                        '''
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:

                    '''
                    LTX-2.5 uses a shared ACTIVE_JOB configuration.
                    LTX-2.5 使用共用 ACTIVE_JOB 參數。
                    '''
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'

            '''
            Clear old output metadata so a failed generation cannot reuse another job's file.
            清除前次輸出狀態，避免本次失敗時誤取其他工作的結果。
            '''
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)

            '''
            Execute a fixed local generation cell, never user-supplied code.
            僅執行 Notebook 內固定的生成程式，不會執行請求傳來的原始碼。
            '''
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):

                '''
                Include YuE2 replay metadata and score with the single MP3.
                將 YuE2 的重播參數與 ABC 樂譜一起提供下載。
                '''
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

'''
Keep one worker and one maintenance thread even after rerunning this cell.
重跑 Cell 仍僅保留一個 worker 和一個清理 thread。
'''
if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()
if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

'''
Close only the previous ngrok tunnel; preserve the running API and its jobs.
只關閉舊的 ngrok 通道，保留正在運作的 API 與所有生成工作。
'''
old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)
NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(Path('/kaggle/working/ngrok'))
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

'''
Always request the auth token again when this cell is rerun.
每次重跑均重新要求輸入 ngrok Authtoken，不顯示內容。
'''
ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')
result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')
del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

'''
Reuse the same API server when this cell is executed again.
重跑本格時沿用同一個 API Server，不中斷生成工作或清空佇列。
'''
server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()
if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:

    '''
    Update the handler on rerun without restarting an active GPU job.
    重跑後更新 API handler，但不重新啟動正在執行的生成工作。
    '''
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')
with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)
NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)
if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])
print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')


## 8. Web API 使用說明

In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request

'''
Request the API schema directly from the local API server.
直接從本機 API Server 查詢支援的參數與模式。
'''
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)
print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本地 requests 範例：06_HeartMuLa_音樂生成_Kaggle_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')